In [2]:
import sklearn, numpy, joblib, scipy
print(sklearn.__version__, numpy.__version__, joblib.__version__, scipy.__version__)

1.6.1 2.1.3 1.6.0 1.16.3


In [3]:
!pip install -q pandas numpy scikit-learn joblib fastapi uvicorn pydantic

In [4]:
import zipfile

with zipfile.ZipFile("news.csv.zip", "r") as zip_ref:
    zip_ref.extractall()

print("Done!")

Done!


In [5]:
import pandas as pd
import numpy as np

# Load your news.csv file
# Expected columns: 'text' and 'label' (0=FAKE, 1=REAL)
df = pd.read_csv('news.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nSample:')
display(df.head())
print('\nLabel distribution:')
print(df['label'].value_counts())

Shape: (6335, 4)

Columns: ['Unnamed: 0', 'title', 'text', 'label']

Sample:


,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL



Label distribution:
label
REAL    3171
FAKE    3164
Name: count, dtype: int64


In [6]:
print('Missing values:')
print(df.isnull().sum())

Missing values:
Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


In [7]:
import re

def wordopt(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)          # remove [brackets]
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'<.*?>+', '', text)            # remove HTML tags
    text = re.sub(r'[^\w\s]', '', text)           # remove punctuation
    text = re.sub(r'\n', ' ', text)               # remove newlines
    text = re.sub(r'\w*\d\w*', '', text)          # remove words with numbers
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra spaces
    return text

# Apply cleaning
df['text_clean'] = df['text'].apply(wordopt)

# Drop rows with empty text after cleaning
df = df[df['text_clean'].str.strip() != '']
df = df.dropna(subset=['text_clean', 'label'])

print(f'Cleaned dataset: {df.shape}')
print('\nSample cleaned text:')
print(df['text_clean'].iloc[0][:200])

Cleaned dataset: (6298, 5)

Sample cleaned text:
daniel greenfield a shillman journalism fellow at the freedom center is a new york writer focusing on radical islam in the final stretch of the election hillary rodham clinton has gone to war with the


In [21]:
from sklearn.model_selection import train_test_split
df['label'].replace({'FAKE': 0, 'REAL': 1}, inplace=True)
X = df['text_clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')
print(f'\nTrain label distribution:\n{y_train.value_counts()}')

Train size: 5038
Test size:  1260

Train label distribution:
label
1    2537
0    2501
Name: count, dtype: int64


/tmp/ipykernel_936/1344163400.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'].replace({'FAKE': 0, 'REAL': 1}, inplace=True)


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_vec = vectorization.fit_transform(X_train)
X_test_vec  = vectorization.transform(X_test)

print(f'Vectorizer fitted')
print(f'Vocabulary size: {len(vectorization.vocabulary_)}')
print(f'Feature matrix:  {X_train_vec.shape}')

Vectorizer fitted
Vocabulary size: 10000
Feature matrix:  (5038, 10000)


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

models = {
    'LR': LogisticRegression(max_iter=1000, random_state=42),
    'DT': DecisionTreeClassifier(random_state=42,max_depth=7,min_samples_leaf=2),
    'GB': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'RF': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

trained = {}
results = {}

for name, model in models.items():
    print(f'Training {name}')
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    trained[name] = model
    results[name] = acc

print('\nAccuracy:')
for name, acc in results.items():
    print(f'   {name}: {acc*100:.2f}%')

Training LR
Training DT
Training GB
Training RF

Accuracy:
   LR: 93.17%
   DT: 80.56%
   GB: 88.65%
   RF: 91.98%


In [24]:
for name, model in trained.items():
    train_acc = model.score(X_train_vec, y_train)
    test_acc = model.score(X_test_vec, y_test)
    print(f'{name}: Train={train_acc*100:.2f}%  Test={test_acc*100:.2f}%')

LR: Train=95.55%  Test=93.17%
DT: Train=85.79%  Test=80.56%
GB: Train=93.25%  Test=88.65%
RF: Train=100.00%  Test=91.98%


In [25]:
print('\nclassification Report LR:')
y_pred_LR = models['LR'].predict(X_test_vec)
print(classification_report(y_test, y_pred_LR, target_names=['FAKE', 'REAL']))


classification Report LR:
              precision    recall  f1-score   support

        FAKE       0.91      0.95      0.93       626
        REAL       0.95      0.91      0.93       634

    accuracy                           0.93      1260
   macro avg       0.93      0.93      0.93      1260
weighted avg       0.93      0.93      0.93      1260



In [26]:
print('\nclassification Report DT:')
y_pred_LR = models['DT'].predict(X_test_vec)
print(classification_report(y_test, y_pred_LR, target_names=['FAKE', 'REAL']))


classification Report DT:
              precision    recall  f1-score   support

        FAKE       0.76      0.89      0.82       626
        REAL       0.87      0.72      0.79       634

    accuracy                           0.81      1260
   macro avg       0.81      0.81      0.80      1260
weighted avg       0.81      0.81      0.80      1260



In [27]:
print('\nclassification Report LR:')
y_pred_LR = models['GB'].predict(X_test_vec)
print(classification_report(y_test, y_pred_LR, target_names=['FAKE', 'REAL']))


classification Report LR:
              precision    recall  f1-score   support

        FAKE       0.87      0.91      0.89       626
        REAL       0.91      0.86      0.88       634

    accuracy                           0.89      1260
   macro avg       0.89      0.89      0.89      1260
weighted avg       0.89      0.89      0.89      1260



In [28]:
print('\nclassification Report LR:')
y_pred_LR = models['RF'].predict(X_test_vec)
print(classification_report(y_test, y_pred_LR, target_names=['FAKE', 'REAL']))


classification Report LR:
              precision    recall  f1-score   support

        FAKE       0.92      0.92      0.92       626
        REAL       0.92      0.92      0.92       634

    accuracy                           0.92      1260
   macro avg       0.92      0.92      0.92      1260
weighted avg       0.92      0.92      0.92      1260



In [29]:
best_model_name = max(results, key=results.get)
best_model = trained[best_model_name]

print(f'best Model: {best_model_name} ({results[best_model_name]*100:.2f}%)')
print('\nclassification Report (Best Model):')
y_pred_best = best_model.predict(X_test_vec)
print(classification_report(y_test, y_pred_best, target_names=['0', '1']))

best Model: LR (93.17%)

classification Report (Best Model):
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       626
           1       0.95      0.91      0.93       634

    accuracy                           0.93      1260
   macro avg       0.93      0.93      0.93      1260
weighted avg       0.93      0.93      0.93      1260



In [35]:
import numpy as np
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

# soft vote, average predicted probabilities
for combo in (['LR', 'RF'], ['LR', 'RF', 'GB'], ['LR', 'DT', 'GB', 'RF']):
    proba = np.mean([trained[n].predict_proba(X_test_vec) for n in combo], axis=0)
    print(f'Soft vote {combo}: {accuracy_score(y_test, proba.argmax(axis=1))*100:.2f}')

# 3. Weighted soft vote
w = {'LR': 3, 'RF': 2, 'GB': 1, 'DT': 0.5}
proba = sum(w[n] * trained[n].predict_proba(X_test_vec) for n in w) / sum(w.values())
print(f'Weighted soft vote:{accuracy_score(y_test, proba.argmax(axis=1))*100:.2f}')

Soft vote ['LR', 'RF']: 93.73
Soft vote ['LR', 'RF', 'GB']: 93.10
Soft vote ['LR', 'DT', 'GB', 'RF']: 89.76
Weighted soft vote:93.57


In [36]:
import os
import joblib

SAVE_DIR = 'FAKE_NEWS_DETECTION'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save all 4 models
joblib.dump(trained['LR'], f'{SAVE_DIR}/lr_model.joblib')
joblib.dump(trained['DT'], f'{SAVE_DIR}/dt_model.joblib')
joblib.dump(trained['GB'], f'{SAVE_DIR}/gb_model.joblib')
joblib.dump(trained['RF'], f'{SAVE_DIR}/rf_model.joblib')

# Save vectorizer
joblib.dump(vectorization, f'{SAVE_DIR}/vectorizer.joblib')

print(f'All models saved to {SAVE_DIR}/')
print('\nFiles saved:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}')
    print(f'{f} -  {size/1024:.1f} KB')

All models saved to FAKE_NEWS_DETECTION/

Files saved:
dt_model.joblib -  12.0 KB
gb_model.joblib -  139.3 KB
lr_model.joblib -  79.0 KB
rf_model.joblib -  9263.9 KB
vectorizer.joblib -  375.2 KB


In [37]:
import os, json
import numpy as np
import joblib
from itertools import combinations
from sklearn.metrics import accuracy_score

SAVE_DIR = 'FAKE_NEWS_DETECTION'
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save all 4 models and the vectorizer
for name in ['LR', 'DT', 'GB', 'RF']:
    joblib.dump(trained[name], f'{SAVE_DIR}/{name.lower()}_model.joblib')
joblib.dump(vectorization, f'{SAVE_DIR}/vectorizer.joblib')

# 2. Compare soft-voting combinations (average predicted probabilities)
proba = {n: trained[n].predict_proba(X_test_vec) for n in trained}
candidates = {}
for r in range(1, 5):
    for combo in combinations(['LR', 'DT', 'GB', 'RF'], r):
        avg = np.mean([proba[n] for n in combo], axis=0)
        candidates[combo] = accuracy_score(y_test, avg.argmax(axis=1))

print('Soft-voting accuracy by combination:')
for combo, acc in sorted(candidates.items(), key=lambda x: -x[1]):
    print(f'  {"+".join(combo):<16} {acc*100:.2f}%')

# 3. Pick the best; on a tie prefer fewer models (simpler)
best_combo = max(candidates, key=lambda c: (round(candidates[c], 4), -len(c)))
print(f'\nSelected ensemble: {"+".join(best_combo)} ({candidates[best_combo]*100:.2f}%)')

# 4. Save the ensemble config for the API
config = {'members': list(best_combo), 'method': 'soft_vote',
          'test_accuracy': round(candidates[best_combo], 4)}
with open(f'{SAVE_DIR}/ensemble_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'\nAll files saved to {SAVE_DIR}/')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'{f} - {os.path.getsize(f"{SAVE_DIR}/{f}")/1024:.1f} KB')

Soft-voting accuracy by combination:
  LR+RF            93.73%
  LR               93.17%
  LR+GB+RF         93.10%
  RF               91.98%
  LR+GB            91.98%
  GB+RF            91.11%
  LR+DT+GB+RF      89.76%
  LR+DT+RF         89.37%
  GB               88.65%
  LR+DT+GB         87.78%
  DT+GB+RF         86.67%
  DT+RF            84.13%
  LR+DT            83.57%
  DT+GB            82.14%
  DT               80.56%

Selected ensemble: LR+RF (93.73%)

All files saved to FAKE_NEWS_DETECTION/
dt_model.joblib - 12.0 KB
ensemble_config.json - 0.1 KB
gb_model.joblib - 139.3 KB
lr_model.joblib - 79.0 KB
rf_model.joblib - 9263.9 KB
vectorizer.joblib - 375.2 KB


In [39]:
app_code = '''
import re
import json

import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI(
    title="Fake News Detection API",
    description="Detects fake news using a soft-voting ensemble of ML models (LR, DT, GB, RF)",
    version="2.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

FAKE_NEWS_DETECTION = "FAKE_NEWS_DETECTION"

# Load all models
lr            = joblib.load(f"{FAKE_NEWS_DETECTION}/lr_model.joblib")
dt            = joblib.load(f"{FAKE_NEWS_DETECTION}/dt_model.joblib")
gb            = joblib.load(f"{FAKE_NEWS_DETECTION}/gb_model.joblib")
rf            = joblib.load(f"{FAKE_NEWS_DETECTION}/rf_model.joblib")
vectorization = joblib.load(f"{FAKE_NEWS_DETECTION}/vectorizer.joblib")

models = {"LR": lr, "DT": dt, "GB": gb, "RF": rf}

# Ensemble members chosen in the notebook
with open(f"{FAKE_NEWS_DETECTION}/ensemble_config.json") as f:
    ENSEMBLE = json.load(f)["members"]

print("All models loaded successfully. Ensemble:", ENSEMBLE)


def wordopt(text: str) -> str:
    """Clean and normalize text."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\\[.*?\\]", "", text)
    text = re.sub(r"https?://\\S+|www\\.\\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[^\\w\\s]", "", text)
    text = re.sub(r"\\n", " ", text)
    text = re.sub(r"\\w*\\d\\w*", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


label_map = {0: "FAKE", 1: "REAL"}


class NewsRequest(BaseModel):
    text: str


class PredictionResponse(BaseModel):
    input_text_preview: str
    model_votes: dict
    overall_verdict: str
    agreement: str
    confidence: str


@app.get("/", include_in_schema=False)
def home():
    return FileResponse("index.html")


@app.post("/predict", response_model=PredictionResponse)
def predict(payload: NewsRequest):
    cleaned = wordopt(payload.text)
    if not cleaned:
        raise HTTPException(status_code=400, detail="Text is empty after cleaning")

    xv = vectorization.transform([cleaned])

    # Individual model predictions (shown in the UI)
    preds = {name: label_map[int(m.predict(xv)[0])] for name, m in models.items()}

    # Soft vote: average predicted probabilities of the selected members
    avg = np.mean([models[n].predict_proba(xv)[0] for n in ENSEMBLE], axis=0)
    idx = int(avg.argmax())

    preview = payload.text[:100] + ("..." if len(payload.text) > 100 else "")

    return {
        "input_text_preview": preview,
        "model_votes": preds,
        "overall_verdict": label_map[idx],
        "agreement": "Ensemble: " + "+".join(ENSEMBLE),
        "confidence": f"{avg[idx] * 100:.0f}%"
    }


@app.get("/health")
def health():
    return {"status": "healthy", "models_loaded": len(models), "ensemble": ENSEMBLE}
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('app.py created!')
!cat app.py

app.py created!

import re
import json

import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI(
    title="Fake News Detection API",
    description="Detects fake news using a soft-voting ensemble of ML models (LR, DT, GB, RF)",
    version="2.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

FAKE_NEWS_DETECTION = "FAKE_NEWS_DETECTION"

# Load all models
lr            = joblib.load(f"{FAKE_NEWS_DETECTION}/lr_model.joblib")
dt            = joblib.load(f"{FAKE_NEWS_DETECTION}/dt_model.joblib")
gb            = joblib.load(f"{FAKE_NEWS_DETECTION}/gb_model.joblib")
rf            = joblib.load(f"{FAKE_NEWS_DETECTION}/rf_model.joblib")
vectorization = joblib.load(f"{FAKE_NEWS_DETECTION}/vectorizer.joblib")

models = {"LR": lr, "DT": dt, "GB": gb, "RF": rf}

# En

In [40]:
dockerfile = '''# ---- Base image ----
FROM python:3.11-slim

WORKDIR /app
ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY index.html .
COPY FAKE_NEWS_DETECTION/ ./FAKE_NEWS_DETECTION/

EXPOSE 8000

CMD ["sh", "-c", "exec uvicorn app:app --host 0.0.0.0 --port ${PORT:-8000}"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

print('dockerfile created!')
!cat Dockerfile

dockerfile created!
# ---- Base image ----
FROM python:3.11-slim

WORKDIR /app
ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY index.html .
COPY FAKE_NEWS_DETECTION/ ./FAKE_NEWS_DETECTION/

EXPOSE 8000

CMD ["sh", "-c", "exec uvicorn app:app --host 0.0.0.0 --port ${PORT:-8000}"]


In [41]:
requirements = '''scikit-learn==1.6.1
numpy==2.1.3
scipy==1.16.3
joblib==1.6.0
fastapi
uvicorn
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('requirements.txt created!')
!cat requirements.txt

requirements.txt created!
scikit-learn==1.6.1
numpy==2.1.3
scipy==1.16.3
joblib==1.6.0
fastapi
uvicorn


In [42]:
dockerignore = '''__pycache__/
*.pyc
*.pyo
*.ipynb
*.zip
*.csv
DATASET/
DATASET.zip
.ipynb_checkpoints/
'''

with open('.dockerignore', 'w') as f:
    f.write(dockerignore)

print('.dockerignore created!')

.dockerignore created!


In [43]:
import os

required_files = [
    'app.py',
    'index.html',
    'Dockerfile',
    'requirements.txt',
    '.dockerignore',
    'FAKE_NEWS_DETECTION/lr_model.joblib',
    'FAKE_NEWS_DETECTION/dt_model.joblib',
    'FAKE_NEWS_DETECTION/gb_model.joblib',
    'FAKE_NEWS_DETECTION/rf_model.joblib',
    'FAKE_NEWS_DETECTION/vectorizer.joblib',
    'FAKE_NEWS_DETECTION/ensemble_config.json',
]

print('File check:')
missing = []
for f in required_files:
    if os.path.exists(f):
        print(f'  OK       {f}  ({os.path.getsize(f) / 1024:.1f} KB)')
    else:
        print(f'  MISSING  {f}')
        missing.append(f)

if not missing:
    print('\nAll files ready for Docker!')
else:
    print('\nMissing files:', ', '.join(missing))

File check:
  OK       app.py  (2.9 KB)
  MISSING  index.html
  OK       Dockerfile  (0.3 KB)
  OK       requirements.txt  (0.1 KB)
  OK       .dockerignore  (0.1 KB)
  OK       FAKE_NEWS_DETECTION/lr_model.joblib  (79.0 KB)
  OK       FAKE_NEWS_DETECTION/dt_model.joblib  (12.0 KB)
  OK       FAKE_NEWS_DETECTION/gb_model.joblib  (139.3 KB)
  OK       FAKE_NEWS_DETECTION/rf_model.joblib  (9263.9 KB)
  OK       FAKE_NEWS_DETECTION/vectorizer.joblib  (375.2 KB)
  OK       FAKE_NEWS_DETECTION/ensemble_config.json  (0.1 KB)

Missing files: index.html


In [45]:
# Zip all required files
!zip -r fake-news-detection_system.zip \
    app.py \
    Dockerfile \
    requirements.txt \
    .dockerignore \
    FAKE_NEWS_DETECTION/

# Show zip size
!du -sh fake-news-detection_system.zip

# Download to your PC
from google.colab import files
files.download('fake-news-detection_system.zip')

print('downloaded')

updating: app.py (deflated 57%)
updating: Dockerfile (deflated 27%)
updating: requirements.txt (deflated 8%)
updating: .dockerignore (deflated 24%)
updating: FAKE_NEWS_DETECTION/ (stored 0%)
updating: FAKE_NEWS_DETECTION/rf_model.joblib (deflated 77%)
updating: FAKE_NEWS_DETECTION/ensemble_config.json (deflated 16%)
updating: FAKE_NEWS_DETECTION/gb_model.joblib (deflated 68%)
updating: FAKE_NEWS_DETECTION/lr_model.joblib (deflated 5%)
updating: FAKE_NEWS_DETECTION/vectorizer.joblib (deflated 70%)
updating: FAKE_NEWS_DETECTION/dt_model.joblib (deflated 65%)
2.4M	fake-news-detection_system.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded
